In [ ]:
import os
from pathlib import Path
import re
import torch as th
from imitation.data import rollout
from imitation.data.types import Trajectory
from imitation.data import rollout
import numpy as np
from imitation.algorithms import bc
import gymnasium as gym
from imitation.data.types import Transitions
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticPolicy
import torch.nn as nn
from sdlarch_rl.utils.utils import get_last_index, GenericCNN
import gc
from IPython import get_ipython
import cv2

rng = np.random.default_rng(0)

demo_path = 'demos-sf6/'

train_path = 'imitation-sf6/'

os.makedirs(train_path, exist_ok=True)


ENT_WEIGHT= 1e-3 # 0 # 1e-4
BATCH_SIZE= 128 # 64 # 128 # 32 # 64 # 128
NUMBER_OF_EPOCH=20
EPOCH_PER_FILE=2 # 3 # 2
MINI_BATCH=64
L2=1e-5
learning_rate=2e-4
buffer_size = 6
# NUMBER_OF_EPOCH = 1

action_space = gym.spaces.MultiBinary(7)

observation_space = gym.spaces.Box(
    low=0,
    high=255,
    shape=(4, 96, 96), # 4 frames 96x96
    dtype=np.uint8,
)


last_index = int(get_last_index(demo_path, "demos", "pt"))
last_index_imitation = int(get_last_index(train_path, "bc_policy", "zip"))

print("last_index: " + str(last_index))
print("Last training session saved: ", f"bc_policy{last_index_imitation}.zip")

# files_index = np.arange(last_index + 1)
files_index = np.arange(last_index + 1)

print("Before trainning files: ", files_index)

class CustomActorCriticPolicy(ActorCriticPolicy):
    def __init__(self, *args, **kwargs):
        super().__init__(
            *args,
            **kwargs,
            features_extractor_class=GenericCNN,
            features_extractor_kwargs=dict(features_dim=256),
            net_arch=[dict(pi=[256, 256], vf=[256, 256])]
        )


policy = CustomActorCriticPolicy(
    observation_space=observation_space,
    action_space=action_space,
    lr_schedule=lambda _: th.finfo(th.float32).max,  # BC control the learning rate
)


th.serialization.add_safe_globals([Trajectory])


bc_trainer = bc.BC(
    observation_space=observation_space,
    action_space=action_space,
    rng=rng,
    ent_weight=ENT_WEIGHT,
    batch_size=BATCH_SIZE,
    policy=policy,
    optimizer_kwargs=dict(lr=learning_rate),
    l2_weight=L2,
    minibatch_size=MINI_BATCH,
)

def concat_transitions(list_of_transitions):
    return Transitions(
        obs=np.concatenate([t.obs for t in list_of_transitions]),
        acts=np.concatenate([t.acts for t in list_of_transitions]),
        next_obs=np.concatenate([t.next_obs for t in list_of_transitions]),
        dones=np.concatenate([t.dones for t in list_of_transitions]),
        infos=np.concatenate([t.infos for t in list_of_transitions]),
    )

def fix_action_format(acts):
    if isinstance(acts, np.ndarray):
        if acts.ndim == 3 and acts.shape[1] == 1:
            acts = acts.squeeze(1)
    return acts

def flip_obs(obs):
    # obs: (T+1, C, H, W)
    return np.flip(obs, axis=-1)

def flip_acts(acts):
    acts = acts.copy()
    acts[:, [2, 3]] = acts[:, [3, 2]]  # swap left/right
    return acts

epoch_count = 0
for e in range(NUMBER_OF_EPOCH):
    np.random.shuffle(files_index)

    epoch_count += 1

    print(f"\n--------------- Epoch: {epoch_count} ------------------\n")

    print("files_index: ", files_index)

    buffer = []

    buffer_files = []

    for i in files_index:
        trajectories = th.load(demo_path + f"demos{i}.pt", weights_only=False)
        fixed_trajectories = []
            
        for traj in trajectories:
            # obs = np.array(traj.obs) if not isinstance(traj.obs, np.ndarray) else traj.obs
            obs = np.array(traj.obs)

            acts = fix_action_format(np.array(traj.acts, dtype=np.float32))

            # Case (T, 1, H, W, C)
            if obs.ndim == 5 and obs.shape[1] == 1:
                obs = obs[:, 0]  # remove dimension 1 → (T, H, W, C)

            # Case (T, C, H, W, 1)
            if obs.ndim == 5 and obs.shape[4] == 1:
                obs = obs.squeeze(-1)
            
            # Case HWC → CHW
            if obs.ndim == 4 and obs.shape[-1] == 4:
                obs = obs.transpose(0, 3, 1, 2)  # (T, 4, 96, 96)

            fixed_trajectories.append(
                Trajectory(
                    obs=obs,
                    acts=acts,
                    infos=traj.infos,
                    terminal=traj.terminal
                )
            )

            # augmentation with flipped data
            flipped_obs = flip_obs(obs)
            flipped_acts = flip_acts(acts)

            fixed_trajectories.append(
                Trajectory(
                    obs=flipped_obs,
                    acts=flipped_acts,
                    infos=traj.infos,
                    terminal=traj.terminal,
                )
            )

        ############### end for loop #######################

        
        np.random.shuffle(fixed_trajectories)
        
        transitions = rollout.flatten_trajectories(fixed_trajectories)

        buffer.append(transitions)
        buffer_files.append(i)

        if len(buffer) == buffer_size:
            merged = concat_transitions(buffer)

            print(f"Processing files: {buffer_files}")

            bc_trainer.set_demonstrations(merged)
            bc_trainer.train(n_epochs=EPOCH_PER_FILE)
            buffer.clear()
            buffer_files.clear()

        del transitions
        del fixed_trajectories
        del trajectories

bc_trainer.policy.save(train_path + f"bc_policy{last_index_imitation + 1}.zip")

gc.collect()
bc_trainer._demonstrations = None
bc_trainer._demonstrations_tensor = None
del bc_trainer

th.cuda.empty_cache()

print("Force cell kernel reset")
get_ipython().kernel.do_shutdown(restart=True)

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


last_index: 8
Last training session saved:  bc_policy10.zip
Before trainning files:  [0 1 2 3 4 5 6 7 8]

--------------- Epoch: 1 ------------------

files_index:  [0 7 8 1 6 5 2 3 4]


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:484: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(


Processing files: [0, 7, 8, 1, 6, 5]


1batch [00:00,  2.63batch/s]

...

0batch [00:00, ?batch/s]

---------------------------------
| batch_size        | 64        |
| bc/               |           |
|    batch          | 0         |
|    ent_loss       | -0.000313 |
|    entropy        | 0.313     |
|    epoch          | 0         |
|    l2_loss        | 0.0162    |
|    l2_norm        | 1.62e+03  |
|    loss           | 0.264     |
|    neglogp        | 0.248     |
|    prob_true_act  | 0.83      |
|    samples_so_far | 128       |
---------------------------------


1000batch [00:17, 59.28batch/s]

---------------------------------
| batch_size        | 64        |
| bc/               |           |
|    batch          | 500       |
|    ent_loss       | -0.000338 |
|    entropy        | 0.338     |
|    epoch          | 0         |
|    l2_loss        | 0.0163    |
|    l2_norm        | 1.63e+03  |
|    loss           | 0.281     |
|    neglogp        | 0.265     |
|    prob_true_act  | 0.821     |
|    samples_so_far | 64128     |
---------------------------------


1752batch [00:29, 58.80batch/s]